In [24]:
import numpy as np
import pandas as pd
import torch
import yaml
from torch.utils.data import DataLoader

from experiment import _fit_propnet
from src.config import Config
from src.data import load_ihdp, make_ihdp_confounded

In [25]:
with open("config/ihdp.yaml") as f:
    cfg = Config.model_validate(yaml.safe_load(f))

train_ds, val_ds, test_ds, ytrain_std = load_ihdp(
    cfg.data.path,
    replication=1,
    train_ratio=cfg.data.train_ratio,
    test_ratio=cfg.data.test_ratio,
)

train_ds, val_ds, test_ds = (
    make_ihdp_confounded(ds, effect=cfg.data.confounder_effect)
    for ds in (train_ds, val_ds, test_ds)
)

In [32]:
torch.manual_seed(cfg.train.seed)

propnet = _fit_propnet(cfg, train_ds, val_ds, test_ds)

train_loader = DataLoader(train_ds, batch_size=cfg.train.batch_size)
device = "cpu"

all_ipw_raw = []
all_ipw_processed = []

with torch.no_grad():
    for batch in train_loader:
        x = batch["x"].to(device)
        a = batch["a"].to(device)

        ipw_raw = propnet.get_importance_weights(x, a)

        ipw_processed = ipw_raw.clamp(0.5, 3)
        ipw_processed = ipw_processed / ipw_processed.mean()

        all_ipw_raw.append(ipw_raw.cpu().numpy())
        all_ipw_processed.append(ipw_processed.cpu().numpy())

In [33]:
conf = train_ds.confounder
weights_raw = np.concatenate(all_ipw_raw)
weights_processed = np.concatenate(all_ipw_processed)

In [34]:
df = pd.DataFrame(
    {"conf": conf, "weights_raw": weights_raw, "weights_processed": weights_processed}
)
df[df["conf"] == 0].describe()

,conf,weights_raw,weights_processed
count,316.0,316.000000,316.000000
mean,0.0,2.010074,1.001451
std,0.0,0.404903,0.190713
min,0.0,1.380688,0.697004
25%,0.0,1.686307,0.842157
50%,0.0,1.927315,0.965880
75%,0.0,2.226723,1.117261
max,0.0,3.878079,1.514470


In [35]:
df[df["conf"] == 1].describe()

,conf,weights_raw,weights_processed
count,373.0,373.000000,373.000000
mean,1.0,2.002142,0.998771
std,0.0,0.414828,0.201117
min,1.0,1.304316,0.658449
25%,1.0,1.661245,0.831089
50%,1.0,1.887980,0.948366
75%,1.0,2.265228,1.135010
max,1.0,3.792044,1.514470
